# Residential Composting Rate by Community District Over Time
**A basic spatiotemporal analysis of NYC DSNY Monthly Tonnage Data**

Source: [DSNY Monthly Tonnage Data](https://data.cityofnewyork.us/City-Government/DSNY-Monthly-Tonnage-Data/ebb7-mvp5/about_data) (dataset id `ebb7-mvp5`)

**Scope:** January 2021 onward.

**Two rates, computed in this notebook (there is no rate column in the raw data):**
1. **Diversion-style rate** (Steps 4-8) = residential organics ÷ everything collected.
2. **True capture rate** (Step 9) = residential organics ÷ compostable material *generated*, using the 2023 Waste Characterization Study (~35.8% of residential waste is compostable).

- **Spatial** dimension = `communitydistrict` · **Temporal** dimension = `month`

> **What counts as "residential organics"?** `resorganicstons` + `leavesorganictons` + `xmastreetons` + `otherorganicstons`; `schoolorganictons` is **excluded** (institutional). Leaves and trees are bundled in, so the rate is seasonal — set `INCLUDE_YARD_WASTE = False` for a food-focused view.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## 1. Pull the data from the Socrata API

No download needed. **Gotcha:** Socrata returns only 1,000 rows unless you raise `$limit`.

In [ ]:
DATASET_ID = "ebb7-mvp5"
url = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.csv?$limit=200000"

df = pd.read_csv(url)
print("Rows, columns:", df.shape)
df.head()

## 2. Define our column groups

In [ ]:
TIME_COL  = "month"
BORO_COL  = "borough"
DIST_COL  = "communitydistrict"

REFUSE = "refusetonscollected"
PAPER  = "papertonscollected"
MGP    = "mgptonscollected"

# Residential organics streams (school organics deliberately excluded)
INCLUDE_YARD_WASTE = True
if INCLUDE_YARD_WASTE:
    ORG_COLS = ["resorganicstons", "leavesorganictons", "xmastreetons", "otherorganicstons"]
else:
    ORG_COLS = ["resorganicstons", "otherorganicstons"]

RECYCLING = [REFUSE, PAPER, MGP]
ALL_STREAMS = RECYCLING + ORG_COLS
print("Numerator (organics):", ORG_COLS)
print("Denominator (all):   ", ALL_STREAMS)

## 3. Clean: coerce numbers, parse the month, and filter to 2021 onward

In [ ]:
# Coerce tonnage columns to numeric
for c in ALL_STREAMS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Parse the month field into a first-of-month timestamp ("YYYY / MM", "YYYY-MM", etc.)
def parse_month(s):
    m = re.search(r"(\d{4})\D+(\d{1,2})", str(s))
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), 1) if m else pd.NaT

df["date"] = df[TIME_COL].map(parse_month)
df = df.dropna(subset=["date"])

# ---- Scope: 2021 onward ----
START = "2021-01-01"
df = df[df["date"] >= START].sort_values("date")

print("Date range:", df["date"].min().date(), "->", df["date"].max().date())
print("Districts: ", df[DIST_COL].nunique())

## 4. Build the streams and the (diversion-style) composting rate

In [ ]:
df["organics"] = df[ORG_COLS].sum(axis=1)
df["total"]    = df[ALL_STREAMS].sum(axis=1)

df["compost_rate"] = np.where(df["total"] > 0,
                              df["organics"] / df["total"] * 100,
                              np.nan)

df[[DIST_COL, "date", "organics", "total", "compost_rate"]].head()

## 5. Temporal view — citywide composting rate over time

Aggregate correctly: **sum tons first, then divide.**

In [ ]:
city = (df.groupby("date")
          .apply(lambda g: g["organics"].sum() / g["total"].sum() * 100,
                 include_groups=False)
          .rename("compost_rate"))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(city.index, city.values, marker="o", ms=4, color="#2e7d32")
ax.set_title("NYC citywide residential composting rate over time (2021+)")
ax.set_ylabel("Composting rate (%)"); ax.set_xlabel("Month")
ax.grid(alpha=0.3)
plt.show()

city.tail(12).round(2)

## 6. Spatial + temporal — one line per borough

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for boro, g in df.groupby(BORO_COL):
    s = (g.groupby("date")
           .apply(lambda x: x["organics"].sum() / x["total"].sum() * 100,
                  include_groups=False))
    ax.plot(s.index, s.values, marker="o", ms=3, label=boro)

ax.set_title("Residential composting rate by borough over time (2021+)")
ax.set_ylabel("Composting rate (%)"); ax.set_xlabel("Month")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 7. The full spatiotemporal matrix — district × month heatmap

In [ ]:
pivot = df.pivot_table(index=DIST_COL, columns="date", values="compost_rate")

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlGn")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=7)
step = max(1, len(pivot.columns) // 24)
ax.set_xticks(range(0, len(pivot.columns), step))
ax.set_xticklabels([d.strftime("%Y-%m") for d in pivot.columns[::step]],
                   rotation=45, ha="right", fontsize=8)
fig.colorbar(im, label="Composting rate (%)")
ax.set_title("Composting rate: community district x month (2021+)")
plt.tight_layout()
plt.show()

## 8. Which districts changed most? (year-over-year)

Compare two **same-month** dates a year apart to control for seasonality. Edit to fit your range.

In [ ]:
def rate_by_district(date_str):
    sub = df[df["date"] == pd.Timestamp(date_str)]
    return (sub.groupby(DIST_COL)
               .apply(lambda g: g["organics"].sum() / g["total"].sum() * 100,
                      include_groups=False))

EARLIER, LATER = "2024-06-01", "2025-06-01"
try:
    change = (rate_by_district(LATER) - rate_by_district(EARLIER)).sort_values(ascending=False).round(2)
    print(f"Composting-rate change (pct points), {EARLIER} -> {LATER}\n")
    print("Biggest increases:\n", change.head(10).to_string())
    print("\nBiggest decreases:\n", change.tail(10).to_string())
except Exception:
    print("Adjust EARLIER/LATER to dates in range:",
          df["date"].min().date(), "->", df["date"].max().date())

## 9. Diversion rate vs. true capture rate, against the policy timeline

The Steps 4-8 rate is a **diversion-style** rate: organics as a share of what was *collected*. But a lot of compostable material never makes it into the organics bin — it's still sitting in the refuse. The **true capture rate** divides by compostable material *generated*, not collected:

$$\text{capture rate} = \frac{\text{organics collected}}{\text{compostable generated}}
= \frac{\text{organics collected}}{f \times \text{total collected}}$$

where **f = 0.358** comes from the 2023 Waste Characterization Study (~35.8% of residential waste is compostable: 21.1% food scraps + 9.0% food-soiled paper + 5.7% yard waste). This is the same method DSNY uses in its Zero Waste Report.

Because capture is just diversion ÷ f, it's a constant rescaling (~2.79×) — but it answers the question people actually mean by "composting rate": *of the compostable material out there, how much got composted?*

We mark your two policy dates:
- **Oct 6, 2024** — residential composting became mandatory
- **Apr 1, 2025** — enforcement and fines went into effect

In [ ]:
COMPOSTABLE_FRACTION = 0.358  # 2023 NYC Waste Characterization Study

POLICY_DATES = {
    "Mandatory (Oct 6 2024)": pd.Timestamp("2024-10-06"),
    "Enforcement (Apr 1 2025)": pd.Timestamp("2025-04-01"),
}

# Citywide, both rates per month
both = df.groupby("date").apply(
    lambda g: pd.Series({
        "diversion_rate": g["organics"].sum() / g["total"].sum() * 100,
        "capture_rate":   g["organics"].sum() / (COMPOSTABLE_FRACTION * g["total"].sum()) * 100,
    }),
    include_groups=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(both.index, both["capture_rate"],   marker="o", ms=4, lw=2,
        color="#1b5e20", label="Capture rate (vs compostable generated)")
ax.plot(both.index, both["diversion_rate"], marker="o", ms=4, lw=2,
        color="#81c784", label="Diversion rate (vs all collected)")

for label, d in POLICY_DATES.items():
    ax.axvline(d, color="#c62828", ls="--", lw=1.5, alpha=0.8)
    ax.text(d, ax.get_ylim()[1]*0.97, " " + label, rotation=90,
            va="top", ha="left", fontsize=9, color="#c62828")

ax.set_title("NYC residential composting: diversion vs. true capture rate (2021+)")
ax.set_ylabel("Rate (%)"); ax.set_xlabel("Month")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Both rates at the policy milestones

The rate in the month each policy took effect, plus the 6 months before and after, so you can see movement around each date.

In [ ]:
def nearest_month(ts):
    """Snap a policy date to the first-of-month bucket used in `both`."""
    return pd.Timestamp(ts.year, ts.month, 1)

summary = (both.round(2)
                .rename(columns={"diversion_rate": "Diversion %",
                                 "capture_rate":   "Capture %"}))

for label, d in POLICY_DATES.items():
    m = nearest_month(d)
    print(f"\n=== {label}  ->  month bucket {m.date()} ===")
    window = summary.loc[(summary.index >= m - pd.DateOffset(months=6)) &
                         (summary.index <= m + pd.DateOffset(months=6))]
    if window.empty:
        print("  (no data in this window)")
    else:
        # mark the policy month with an arrow
        disp = window.copy()
        disp.index = [("-> " if idx == m else "   ") + idx.strftime("%Y-%m")
                      for idx in window.index]
        print(disp.to_string())

### District-level capture rate at each policy date

Capture rate per community district in the month each policy took effect — so you can see the spatial spread, not just the citywide average.

In [ ]:
def district_rates_at(month_ts):
    sub = df[df["date"] == month_ts]
    if sub.empty:
        return pd.DataFrame()
    g = sub.groupby(DIST_COL).apply(
        lambda x: pd.Series({
            "diversion_%": x["organics"].sum() / x["total"].sum() * 100,
            "capture_%":   x["organics"].sum() / (COMPOSTABLE_FRACTION * x["total"].sum()) * 100,
        }), include_groups=False)
    return g.sort_values("capture_%", ascending=False).round(2)

for label, d in POLICY_DATES.items():
    m = pd.Timestamp(d.year, d.month, 1)
    print(f"\n=== {label}  (month {m.date()}) — top & bottom 5 districts by capture rate ===")
    t = district_rates_at(m)
    if t.empty:
        print("  (no data for this month)")
    else:
        print("Highest:\n", t.head(5).to_string())
        print("Lowest:\n",  t.tail(5).to_string())

## Caveats & next steps

**Caveats**
- **"Organics" ≠ pure food composting** — it bundles yard waste and Christmas trees, hence seasonal spikes. Toggle `INCLUDE_YARD_WASTE = False` in Step 2 for a food-focused view.
- **Capture rate uses a single citywide fraction (0.358).** The 2023 study found compostable share varies by neighborhood density/income, but it did **not** publish an organics breakdown by district (curbside organics wasn't citywide during sampling). So the per-district capture rates assume the citywide 35.8% applies everywhere — treat district capture numbers as approximate.
- **Diversion-style rate** is exact from this data; **capture rate** is a modeled estimate layered on top.
- **Geography** is the sanitation community district — operationally defined.

**Next steps**
1. Join a community-district shapefile (GeoPandas) for a choropleth map.
2. If DSNY publishes a post-2024 characterization study, swap in district-specific compostable fractions.
3. Add a 3-month rolling average to smooth the seasonal sawtooth and see the underlying trend across your two policy dates.